# Optimal NIRCam pointing offset

In this notebook, we will search the optimal target position for our NIRCam imaging observations.
We want to search for close companions so what we are looking for is a small detector patch (roughly 70x70 pixels)
where we can place the science target. We still use the FULL detector array to search for wide companions.

We also use primary dithers, so ideally we would want all our primary dither positions to be free of bad pixels.

And finally, we are carrying short wavelength observations in parallel,
so we want to make sure these images are also as free as possible of bad pixels,
and we want to make sure that the target does not fall between two detectors.

## Downloading an observation

We will download a processed NIRCam observation from MAST to access a DQ map on the detector.
We use an observation from the GO 2473 cycle 1 program (P.I. Albert) as they are similar to the ones we will carry.

Let us start by only downloading the long wavelength stage 2 product.

In [ ]:
from pathlib import Path
from astroquery.mast import Observations

x, y = 1282, 825
uri_lw = "mast:JWST/product/jw02473052001_02101_00004_nrcblong_cal.fits"
data_path = Path("./data")
data_path.mkdir(exist_ok=True)
path_lw = data_path / uri_lw.split("/")[-1]

Observations.download_file(uri_lw, local_path=path_lw);

## Looking at the metadata

Let's take a quick look at the science data before we proceed to our pointing optimisation.
Here are all the extensions in a JWST stage 2 image.

In [ ]:
from astropy.io import fits

hdul = fits.open(path_lw)
hdul.info()

We can extract the relevant extensions and headers, and check when the data was processed.

In [ ]:
hdr = hdul[0].header
img = hdul["SCI"].data
dq = hdul["DQ"].data

created_date = hdr["DATE"]
crds_ver = hdr["CRDS_VER"]
crds_ctx = hdr["CRDS_CTX"]
print(f"The file was processed on {created_date} with CRDS version {crds_ver} and context {crds_ctx}")

The file was processed not too long ago so the DQ map should be up to date.

## Looking at the science image

Let's display the science image before we move on to the DQ map.

In [ ]:
from matplotlib import rcParams
import matplotlib.pyplot as plt

rcParams["image.origin"] = "lower"

plt.imshow(img, norm="symlog")
plt.plot(x, y, "r*", label="Target position")
plt.title(path_lw.name)
plt.xlabel("X [pix]")
plt.ylabel("Y [pix]")
plt.colorbar(label="DN/s")
plt.legend()
plt.show()

And let's take a zoomed-in look at our target

In [ ]:
img_size = 70
img_hs = img_size // 2
img_crop = img[y - img_hs: y + img_hs, x - img_hs: x + img_hs]

plt.imshow(img_crop, norm="symlog")
plt.title(f"{path_lw.name} zoomed")
plt.xlabel("X [pix]")
plt.ylabel("Y [pix]")
plt.colorbar(label="DN/s")
plt.show()

## Searching for optimal regions

To avoid messing with the DQ extension, we can just do a nan mask on the images.
(I checked beforehand that this was the same as doing a mask of the `DO_NOT_USE` pixels from the DQ array.

In [ ]:
import numpy as np
dq_mask = np.isnan(img)

What we want to do next is scan all 70x70 patches in the image and find those with the least bad pixels.

### Scipy implementation

Here is an implementation with scipy using a uniform filter


In [ ]:
from scipy.ndimage import uniform_filter

# Average of dq_mask in 70x70 region centered on each pixel
# weighted_mask = dq_mask * img
weighted_mask = dq_mask
dq_filtered = uniform_filter(weighted_mask.astype(float), size=img_size, mode="constant")

# Convert average to count per region
dq_count = dq_filtered * img_size**2

full_size = dq_mask.shape[0]
slice = np.s_[img_hs:full_size-img_size+img_hs+1]
# dq_count_valid = dq_count[slice,slice]

flat_dq_count = dq_count.flatten()

overlap_ok = False
if overlap_ok: 
    n_top = 10
    sorted_idx = np.argsort(flat_dq_count)[:n_top]
    best_x, best_y = np.unravel_index(sorted_idx, dq_count.shape)
else:
    # Select non-overlapping positions
    selected = []
    sorted_idx = np.argsort(flat_dq_count)
    all_rows, all_cols = np.unravel_index(sorted_idx, dq_count.shape)
    for i in range(len(sorted_idx)):
        row, col = int(all_rows[i]), int(all_cols[i])
        weighted_sum = float(flat_dq_count[sorted_idx[i]])

        # Check if this position overlaps with any already selected
        overlaps = False
        for sel_row, sel_col, _ in selected:
            # Two windows overlap if they're within window_size of each other
            if abs(row - sel_row) < img_size and abs(col - sel_col) < img_size:
                overlaps = True
                break

        # If no overlap, add to results
        if not overlaps:
            selected.append((row, col, weighted_sum))

            # Stop once we have enough
            if len(selected) >= n_top:
                break

    best_y = [s[0] for s in selected]
    best_x = [s[1] for s in selected]

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
axs[0].imshow(dq_mask)
im = axs[1].imshow(dq_count)
axs[1].plot(best_x, best_y, "r*")
fig.colorbar(im)
plt.show()

In [ ]:
fig, axs = plt.subplots(1, n_top, figsize=(20, 5))
for i, s in enumerate(selected):
    y, x = s[:2]
    region = img[y - img_hs: y + img_hs, x - img_hs: x + img_hs]
    nan_count = np.sum(np.isnan(region))
    axs[i].imshow(region, norm="symlog")
    axs[i].set_title(nan_count)
plt.show()